## Workflow overview

The figure below summarizes the end-to-end workflow from cell-based perturbations to phenotype modeling. After measuring multimodal phenotypic readouts and organizing them in an AnnData object, UniPert generates unified perturbagen embeddings across genetic and chemical modalities. These embeddings can be used together with unperturbed profiles as inputs to downstream perturbation-effect predictors.

<img src="https://github.com/user-attachments/assets/bfc669c1-4d41-479b-9da3-f345a18623ef" alt="Workflow overview: UniPert representations in AnnData" width="850">

## Generating UniPert representations for perturb anndata

In this tutorial, we will introduce how to generate perturbagen embeddings using UniPert for a given perturbation `AnnData` file:

* The `UniPert representations` will be formatted as a `dict` and stored in the AnnData object under the key `adata.uns['UniPert_reps']`.
  
* The `invalid or unretrieved perturbagens` will be formatted as a `list` and stored in the AnnData object under the key `adata.uns['invalid_ptbgs']`.
  
We use 2 example perturbation adata from [scPerturb database](https://www.sanderlab.org/scPerturb/datavzrd/scPerturb_vzrd_v2/dataset_info/index_1.html) to show the generating process:

  1. [Example 1: Genetic Perturbation Adata](#Genetic-Perturbation-Adata)

  2. [Example 2: Chemical Perturbation Adata](#Chemical-Perturbation-Adata)




## Prepare example pert adata

Define the download function to get perturbation adata file from [scPerturb database](https://www.sanderlab.org/scPerturb/datavzrd/scPerturb_vzrd_v2/dataset_info/index_1.html).

In [1]:
import os
import requests
from lamin_utils import logger

def download_file(url: str, folder_path: str):
    """
    Download file from url to folder_path
    """
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    file_name = url.split('/')[-1]
    file_path = os.path.join(folder_path, file_name)
    # check if file already exists
    if os.path.exists(file_path):
        logger.info(f"{file_name} already exists.")
        return file_path
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  
        with open(file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        logger.download(f"{file_name} download.")
        return file_path
    except requests.exceptions.RequestException as e:
        logger.error(f"Download failed: {e}")
        return None

## Prepare UniPert model

In [2]:
# import sys
# sys.path.append('../')
from unipert import UniPert

unipert = UniPert()

💡 CUDA is not available. Using CPU instead.
💡 Building UniPert model...
✅ ESM2 model loaded.
✅ Reference ESM2 embedding loaded.
✅ UniPert model architecture initialized.
✅ Pretrained UniPert model loaded.
✅ Reference target graph prepared.
✅ Model loaded and initialized.


## Examples

### Genetic Perturbation Adata

In [3]:
scperturb_url = 'https://zenodo.org/record/10044268/files/PapalexiSatija2021_eccite_arrayed_RNA.h5ad'
demo_data_path = '../demo_data/'  
file_path = download_file(scperturb_url, demo_data_path)

💡 PapalexiSatija2021_eccite_arrayed_RNA.h5ad already exists.


In [4]:
import anndata as ad
adata = ad.read_h5ad(file_path, 'r')
adata

AnnData object with n_obs × n_vars = 8984 × 16826 backed at '../demo_data/PapalexiSatija2021_eccite_arrayed_RNA.h5ad'
    obs: 'perturbation', 'hto', 'guide_id', 'hto_barcode', 'gdo_barcode', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'ensembl_id', 'ncounts', 'ncells'

In [5]:
adata.obs['perturbation'].value_counts()

control     2009
ETV7        1789
IRF1         994
ATF2         794
IRF7         750
MARCH8       723
IFNGR1       701
STAT2        576
CAV1         409
PDL1         235
IFNGR2         4
STAT5A         0
SPI1           0
STAT3          0
TNFRSF14       0
UBE2L6         0
STAT1          0
NFKBIA         0
SMAD4          0
POU2F2         0
PDCD1LG2       0
BRD4           0
JAK2           0
CUL3           0
CMTM6          0
CD86           0
eGFP           0
Name: perturbation, dtype: int64

In [6]:
unipert.encode_anndata_perturbations(
    adata=adata,
    perturbation_columns=['perturbation'],
    perturbation_types=['genetic'],
    control_key='control',
    return_results=False
)

💡 19187 reference targets encoded.
💡 Encoding 10 genetic perturbagens with UniPert...


100%|██████████| 10/10 [00:11<00:00,  1.14s/it]

💡 Building reference-custom target graph from /Users/liyiming/Library/CloudStorage/OneDrive-csu.edu.cn/Research_Project/c-UniPert/code/git-version/UniPert/data/custom_target_seq.fasta ...
💡 Computing similarities between /Users/liyiming/Library/CloudStorage/OneDrive-csu.edu.cn/Research_Project/c-UniPert/code/git-version/UniPert/data/custom_target_seq.fasta and reference sequences...


createdb /Users/liyiming/Library/CloudStorage/OneDrive-csu.edu.cn/Research_Project/c-UniPert/code/git-version/UniPert/data/custom_target_seq.fasta /Users/liyiming/Library/CloudStorage/OneDrive-csu.edu.cn/Research_Project/c-UniPert/code/git-version/UniPert/mmseqs_storage/query 

MMseqs Version:                    	18.8cc5c
Database type                      	0
Shuffle input database             	true
Createdb mode                      	0
Write lookup file                  	1
Offset of numeric ids              	0
Threads                            	8
Compressed                         	0
Mask residues                      	0
Mask residues probability          	0.9
Mask lower case residues           	0
Mask lower letter repeating N times	0
Use GPU                            	0
Verbosity                          	3

Converting sequences
[
Time for merging to query_h: 0h 0m 0s 78ms
Time for merging to query: 0h 0m 0s 9ms
Database type: Aminoacid
Time for processing: 0h 0m 0s 111ms
Create di

100%|██████████| 2/2 [00:01<00:00,  1.21it/s]


✅ UniPert representations generated!
💡 10 perturbagens' UniPert representations saved to adata.uns['UniPert_reps']


In [7]:
adata.uns['UniPert_reps'].keys(), adata.uns['invalid_ptbgs']

(dict_keys(['STAT2', 'IRF7', 'ETV7', 'CAV1', 'ATF2', 'IFNGR2', 'IFNGR1', 'IRF1', 'PDL1', 'MARCH8']),
 [])

### Chemical Perturbation Adata

In [8]:
scperturb_url = 'https://zenodo.org/record/10044268/files/SrivatsanTrapnell2020_sciplex3.h5ad'
demo_data_path = '../demo_data/'  
file_name = download_file(scperturb_url, demo_data_path)

💡 SrivatsanTrapnell2020_sciplex3.h5ad already exists.


In [9]:
import anndata as ad

adata = ad.read_h5ad(file_name, 'r')
adata

AnnData object with n_obs × n_vars = 799317 × 110984 backed at '../demo_data/SrivatsanTrapnell2020_sciplex3.h5ad'
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'ngenes', 'percent_mito', 'percent_ribo', 'nperts', 'chembl-ID'
    var: 'ensembl_id', 'ncounts', 'ncells'

In [10]:
adata.obs['perturbation'].value_counts()

control                              17578
Ellagic acid                          6257
Divalproex Sodium                     6203
Ruxolitinib (INCB018424)              6143
MC1568                                6126
                                     ...  
Alvespimycin (17-DMAG) HCl            2089
Patupilone (EPO906, Epothilone B)     1822
Flavopiridol HCl                      1729
Epothilone A                          1426
YM155 (Sepantronium Bromide)          1007
Name: perturbation, Length: 189, dtype: int64

In [12]:
unipert.encode_anndata_perturbations(
    adata=adata,
    perturbation_columns=['perturbation'],
    perturbation_types=['chemical'],
    control_key='control',
    return_results=False
)

💡 Encoding 188 chemcial perturbagens with UniPert...
💡 Retrievaling SMILES for chemical perturbagens...


 14%|█▍        | 26/188 [00:23<02:20,  1.15it/s]

Unable to retrieve SMILES for query compound name: Ivosidenib (AG-120)


 18%|█▊        | 33/188 [00:29<02:10,  1.19it/s]

Unable to retrieve SMILES for query compound name: Dacinostat (LAQ824)


...

Unable to retrieve SMILES for query compound name: Bisindolylmaleimide IX (Ro 31-8220 Mesylate)


 63%|██████▎   | 118/188 [01:43<01:01,  1.14it/s]

Unable to retrieve SMILES for query compound name: Ruxolitinib (INCB018424)


100%|██████████| 181/181 [00:00<00:00, 1507.84it/s]

✅ UniPert representations generated!
💡 181 perturbagens' UniPert representations saved to adata.uns['UniPert_reps']
❗ 7 perturbagens can not be repersentated and saved to adata.uns['invalid_ptbgs']: 
['Ivosidenib (AG-120)', 'Dacinostat (LAQ824)', 'Glesatinib?(MGCD265)', 'Ki16425', 'Tucidinostat (Chidamide)', 'Bisindolylmaleimide IX (Ro 31-8220 Mesylate)', 'Ruxolitinib (INCB018424)']


In [ ]:
# Try again to retrieve perturbagens not successfully retrieved before

# unipert.encode_anndata_perturbations(
#     adata=adata,
#     perturbation_columns=['perturbation'],
#     perturbation_types=['chemical'],
#     control_key='control',
#     return_results=False
# )

In [12]:
len(adata.uns['UniPert_reps'])

183

In [13]:
adata.uns['invalid_ptbgs']

['Dacinostat (LAQ824)',
 'Bisindolylmaleimide IX (Ro 31-8220 Mesylate)',
 'Ivosidenib (AG-120)',
 'Glesatinib?(MGCD265)',
 'Tucidinostat (Chidamide)']